In [ ]:
# ==============================
# 1) Importar bibliotecas
# ==============================
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import tkinter as tk
from tkinter import messagebox
import matplotlib.pyplot as plt

# ==============================
# 2) Carregar Dataset
# ==============================
# Nome do arquivo Excel
DATASET_PATH = "dataset_compressor_balanceado.xlsx"

# Carregar planilha
df = pd.read_excel(DATASET_PATH)

print("🔹 Dataset carregado com sucesso!")
print(df.head())

# ==============================
# 3) Definir Features e Labels
# ==============================
FEATURES = [
    "pressao_entrada_bar", "pressao_saida_bar", "temp_ar_saida_c",
    "corrente_motor_a", "vazao_m3h", "vibracao_rms_mms",
    "delta_p", "temp_motor_c", "horas_operacao", "nivel_oleo_perc"
]

LABEL = "condicao"   # coluna de saída do Excel (falha/normal/etc.)

X = df[FEATURES].values
y = df[LABEL].values

# ==============================
# 4) Pré-processamento
# ==============================
# Normalizar os valores
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Codificar os labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Salvar scaler e encoder para uso posterior na interface
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le, "label_encoder.pkl")

# Separar treino/validação
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# ==============================
# 5) Construir Rede Neural
# ==============================
model = Sequential([
    Dense(64, activation="relu", input_shape=(len(FEATURES),)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(len(le.classes_), activation="softmax")  # saída multi-classe
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Early stopping para evitar overfitting
es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

# Treinar modelo
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[es],
    verbose=1
)

# Salvar modelo
MODEL_PATH = "modelo_compressor_excel.h5"
model.save(MODEL_PATH)

print("✅ Modelo treinado e salvo com sucesso!")

# ==============================
# 6) Gráficos de Acurácia e Perda
# ==============================
plt.figure(figsize=(12,5))

# Gráfico de acurácia
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Acurácia Treino')
plt.plot(history.history['val_accuracy'], label='Acurácia Validação')
plt.title('Evolução da Acurácia')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.legend()
plt.grid(True)

# Gráfico de perda
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Perda Treino')
plt.plot(history.history['val_loss'], label='Perda Validação')
plt.title('Evolução da Perda')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ==============================
# 7) Interface Tkinter
# ==============================
HORIZON_HOURS = 24  # previsão de falha em 24h

def prever_falha(valores_dict):
    # Carregar objetos
    model = tf.keras.models.load_model(MODEL_PATH)
    scaler = joblib.load("scaler.pkl")
    le = joblib.load("label_encoder.pkl")

    # Transformar entradas
    valores = [float(valores_dict[f]) for f in FEATURES]
    entrada = scaler.transform([valores])

    # Fazer previsão
    probs = model.predict(entrada)[0]
    pred_idx = np.argmax(probs)
    pred_label = le.inverse_transform([pred_idx])[0]

    # Montar mensagem
    msg = f"⚠️ Probabilidade de falha em até {HORIZON_HOURS}h:\n"
    msg += f">>> {probs[pred_idx]*100:.1f}% de chance de '{pred_label}'\n\n"
    msg += "Detalhes por condição:\n"
    for lbl, p in zip(le.classes_, probs):
        msg += f" - {lbl}: {p*100:.1f}%\n"

    return msg

def criar_interface():
    root = tk.Tk()
    root.title("Previsão de Falhas - Compressor")
    entries = {}

    # Criar campos para entrada
    for i, feature in enumerate(FEATURES):
        tk.Label(root, text=feature).grid(row=i, column=0, padx=5, pady=5, sticky="w")
        entry = tk.Entry(root)
        entry.grid(row=i, column=1, padx=5, pady=5)
        entries[feature] = entry

    # Botão de previsão
    def on_prever():
        try:
            valores_dict = {f: entries[f].get() for f in FEATURES}
            if any(v == "" for v in valores_dict.values()):
                messagebox.showerror("Erro", "Preencha todos os campos.")
                return

            resultado = prever_falha(valores_dict)
            messagebox.showinfo("Resultado da Previsão", resultado)

        except Exception as e:
            messagebox.showerror("Erro", str(e))

    btn = tk.Button(root, text="Prever", command=on_prever, bg="green", fg="white")
    btn.grid(row=len(FEATURES), column=0, columnspan=2, pady=10)

    root.mainloop()

# Executar interface
if __name__ == "__main__":
    criar_interface()